# Physics-Informed Diffusion-PINN MRI Reconstruction
## Notebook 03 — Zero-Filled Baseline Reconstruction (4× & 8× Acceleration)

This notebook establishes the quantitative and qualitative baseline for accelerated MRI reconstruction using **Zero-Filled Inverse FFT Reconstruction** under 4× and 8× 1D Cartesian undersampling.

### Pipeline:
$$\text{Full K-space } \mathbf{y}_{\text{full}} \xrightarrow{\mathbf{M}} \text{Undersampled K-space } \mathbf{y}_{\text{us}} = \mathbf{M} \odot \mathbf{y}_{\text{full}} \xrightarrow{\mathcal{F}^{-1}} \text{Zero-Filled Recon } \mathbf{x}_{\text{zf}}$$

Quantitative metrics evaluated:
- **PSNR**: Peak Signal-to-Noise Ratio (dB)
- **SSIM**: Structural Similarity Index Measure
- **NMSE**: Normalized Mean Squared Error
- **K-Space Error**: Relative $\ell_2$ norm error at acquired sample coordinates

In [ ]:
# STEP 1 — Setup Environment & Imports
import os
import sys
import json
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import torch
import yaml

# Add repository root directory to sys.path
sys.path.insert(0, os.getcwd())

from src.physics.fft import kspace_to_image_torch, image_to_kspace_torch
from src.data.preprocessing import complex_abs, normalize_zero_one
from src.data.undersampling import RandomMaskFunc
from src.evaluation.metrics import evaluate_reconstruction
from src.visualization.reconstructions import plot_zero_filled_comparison
from src.visualization.error_maps import compute_absolute_error_map

# Load base configuration
with open("configs/base.yaml", "r") as f:
    config = yaml.safe_load(f)

data_dir = Path(config.get("data", {}).get("data_dir", "/kaggle/working/data/fastmri"))
if not data_dir.exists():
    data_dir = Path("data/fastmri")

print(f"Data Directory: {data_dir.resolve()}")

In [ ]:
# STEP 2 — Discover Data and Verify HDF5 File Structure
h5_files = sorted(list(data_dir.glob("*.h5")) + list(data_dir.glob("*.hdf5")))
if not h5_files:
    raise FileNotFoundError(f"No HDF5 files found in {data_dir}. Ensure dataset downloader has run.")

sample_file = h5_files[0]
print(f"Dataset files available : {len(h5_files)}")
print(f"Selecting sample file   : {sample_file.name}")

with h5py.File(sample_file, "r") as hf:
    kspace_shape = hf["kspace"].shape
    print(f"HDF5 Keys               : {list(hf.keys())}")
    print(f"K-space Shape           : {kspace_shape} (slices, height, width)")

In [ ]:
# STEP 3 — Initialize Undersampling Mask Generators
seed = config.get("experiment", {}).get("seed", 42)

# 4x Acceleration (8% center fraction)
mask_func_4x = RandomMaskFunc(center_fractions=[0.08], accelerations=[4], seed=seed)

# 8x Acceleration (4% center fraction)
mask_func_8x = RandomMaskFunc(center_fractions=[0.04], accelerations=[8], seed=seed)

print(f"Mask Generators Initialized (Seed: {seed})")

In [ ]:
# STEP 4 — Run Zero-Filled Baseline Evaluation Loop
# Select a lightweight subset of files/slices for fast Kaggle execution
MAX_FILES = min(5, len(h5_files))
SLICES_PER_FILE = 2

results_list = []
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/logs", exist_ok=True)

print(f"Starting zero-filled evaluation on {MAX_FILES} files ({SLICES_PER_FILE} slices per file)...")

for f_idx in range(MAX_FILES):
    file_path = h5_files[f_idx]
    with h5py.File(file_path, "r") as hf:
        total_slices = hf["kspace"].shape[0]
        start_slice = max(0, (total_slices - SLICES_PER_FILE) // 2)
        
        for s_idx in range(start_slice, start_slice + SLICES_PER_FILE):
            raw_kspace = hf["kspace"][s_idx]
            kspace_tensor = torch.from_numpy(raw_kspace)
            
            # Compute fully sampled ground-truth image via IFFT
            gt_complex = kspace_to_image_torch(kspace_tensor)
            gt_mag = complex_abs(gt_complex).numpy()
            gt_norm = normalize_zero_one(gt_mag)
            
            # Perform evaluation for 4x and 8x acceleration
            for accel, mask_func in [(4, mask_func_4x), (8, mask_func_8x)]:
                mask = mask_func(kspace_tensor.shape, seed=seed + s_idx)
                y_undersampled = mask * kspace_tensor
                
                # Zero-Filled Inverse FFT Reconstruction
                zf_complex = kspace_to_image_torch(y_undersampled)
                zf_mag = complex_abs(zf_complex).numpy()
                zf_norm = normalize_zero_one(zf_mag)
                
                # Metrics
                metrics = evaluate_reconstruction(
                    gt=gt_norm,
                    pred=zf_norm,
                    y_measured=y_undersampled,
                    x_pred_complex=zf_complex,
                    mask=mask
                )
                
                record = {
                    "filename": file_path.name,
                    "slice_idx": s_idx,
                    "acceleration": accel,
                    "psnr": metrics["psnr"],
                    "ssim": metrics["ssim"],
                    "nmse": metrics["nmse"],
                    "kspace_error": metrics["kspace_error"]
                }
                results_list.append(record)
                
                # Save visualization figure for the first sample slice
                if f_idx == 0 and s_idx == start_slice:
                    error_map = compute_absolute_error_map(gt_norm, zf_norm)
                    mask_np = mask.squeeze().numpy()
                    y_np = y_undersampled.numpy()
                    fig_path = f"results/figures/zero_filled_accel{accel}_sample.png"
                    plot_zero_filled_comparison(
                        gt=gt_norm,
                        mask=mask_np,
                        undersampled_kspace=y_np,
                        zf_reconstruction=zf_norm,
                        error_map=error_map,
                        metrics=metrics,
                        accel_factor=accel,
                        save_path=fig_path
                    )
                    print(f"Saved visual figure ({accel}x) to: {fig_path}")

In [ ]:
# STEP 5 — Aggregate Quantitative Results & Export Summary Tables
df_results = pd.DataFrame(results_list)
csv_path = "results/tables/zero_filled_baseline_metrics.csv"
df_results.to_csv(csv_path, index=False)
print(f"Detailed metrics saved to: {csv_path}")

summary = df_results.groupby("acceleration")[["psnr", "ssim", "nmse", "kspace_error"]].agg(["mean", "std"])
print("
========================================")
print("ZERO-FILLED BASELINE METRICS SUMMARY")
print("========================================")
print(summary)
print("========================================")

# Export JSON log summary
log_summary = {}
for accel in [4, 8]:
    sub_df = df_results[df_results["acceleration"] == accel]
    log_summary[f"{accel}x"] = {
        "psnr_mean": float(sub_df["psnr"].mean()),
        "psnr_std": float(sub_df["psnr"].std()),
        "ssim_mean": float(sub_df["ssim"].mean()),
        "ssim_std": float(sub_df["ssim"].std()),
        "nmse_mean": float(sub_df["nmse"].mean()),
        "nmse_std": float(sub_df["nmse"].std()),
        "kspace_error_mean": float(sub_df["kspace_error"].mean()),
        "kspace_error_std": float(sub_df["kspace_error"].std()),
    }

json_log_path = "results/logs/zero_filled_experiment.json"
with open(json_log_path, "w") as f:
    json.dump(log_summary, f, indent=2)
print(f"Log summary saved to: {json_log_path}")

In [ ]:
# STEP 6 — Verification Banner
print("========================================")
print("ZERO-FILLED BASELINE COMPLETED")
print("========================================")
print("Visual Figures  : Saved under results/figures/")
print("Metrics Table   : Saved to results/tables/zero_filled_baseline_metrics.csv")
print("JSON Log        : Saved to results/logs/zero_filled_experiment.json")
print("----------------------------------------")
print("Next Phase:")
print("  Phase 9: U-Net Baseline Architecture Implementation & Training")
print("========================================")